# LeetCode #99 — Recover Binary Search Tree: tìm hai nút bị đổi chỗ qua inorder

**Mức độ:** Medium · **Chủ đề:** Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def recoverTree(self, root: Optional[TreeNode]) -> None:
        """
        Do not return anything, modify root in-place instead.
        """
```

Mục tiêu của notebook:

1. Dùng tính chất "inorder của BST tăng dần" (#98) để **tìm chỗ sai**.
2. Phân biệt hai trường hợp: hai nút bị đổi **kề nhau** (1 chỗ giảm) hay **xa nhau** (2 chỗ giảm).
3. Lời giải `O(h)` bộ nhớ bằng stack.
4. Follow-up `O(1)` bộ nhớ: **Morris traversal**.

# 1. Bài toán là gì?

Cho `root` của một BST, trong đó **đúng hai nút** đã bị **đổi giá trị** cho nhau do nhầm lẫn. Khôi phục cây **mà không thay đổi cấu trúc**.

**Constraints (theo đề):**

- Số nút trong khoảng `[2, 1000]`.
- `-2^31 <= Node.val <= 2^31 - 1`

**Follow up (theo đề):** lời giải `O(n)` bộ nhớ khá đơn giản. Bạn có nghĩ ra lời giải bộ nhớ **hằng** `O(1)` không?

---

## Ví dụ

```text
    1              3
   /              /
  3      ->      1         (1 và 3 bị đổi chỗ)
   \              \
    2              2

    3              2
   / \            / \
  1   4   ->     1   4     (2 và 3 bị đổi chỗ)
     /              /
    2              3
```

LeetCode: `[1,3,null,null,2]` → `[3,1,null,null,2]`; `[3,1,4,null,null,2]` → `[2,1,4,null,null,3]`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Cây nhị phân tìm kiếm (BST)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [7]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [8]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.3 Áp dụng vào bài này

Duyệt **inorder** của BST phải tăng dần; hai nút bị đổi chỗ tạo ra một hoặc hai **chỗ giảm** trong dãy inorder. Nút sai thứ nhất là phần tử lớn ở chỗ giảm đầu, nút sai thứ hai là phần tử nhỏ ở chỗ giảm cuối; đổi lại giá trị của chúng.

# 3. Nền tảng

In [9]:
from typing import List, Optional
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def serialize(root: Optional[TreeNode]) -> list:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def inorder_list(root):
    return inorder_list(root.left) + [root.val] + inorder_list(root.right) if root else []

print(inorder_list(build_tree([1, 3, None, None, 2])))
print(inorder_list(build_tree([3, 1, 4, None, None, 2])))

[3, 2, 1]
[1, 3, 2, 4]


# 4. Quan sát: đổi chỗ hai phần tử trong một dãy tăng

Inorder của BST đúng là dãy **tăng nghiêm ngặt**. Đổi chỗ hai giá trị → xuất hiện các **"chỗ giảm"** (cặp liền kề `a[i] > a[i+1]`):

```text
Đổi hai phần tử XA nhau:   1 2 [6] 4 5 [3] 7   -> 2 chỗ giảm: (6,4) và (5,3)
                                                 sai: phần tử ĐẦU của chỗ giảm 1 (6)
                                                      và phần tử SAU của chỗ giảm 2 (3)
Đổi hai phần tử KỀ nhau:   1 2 [4][3] 5 6 7      -> 1 chỗ giảm: (4,3)
                                                 sai: cả hai phần tử của chỗ giảm đó
```

Quy tắc chung: gọi `first` = phần tử **lớn** của chỗ giảm **đầu tiên**, `second` = phần tử **nhỏ** của chỗ giảm **cuối cùng**. Đổi lại giá trị của `first` và `second`.

In [10]:
def find_swapped(arr: List[int]):
    first = second = None
    for i in range(len(arr) - 1):
        if arr[i] > arr[i + 1]:
            if first is None:
                first = i                    # phần tử lớn ở chỗ giảm đầu
            second = i + 1                   # phần tử nhỏ ở chỗ giảm (cập nhật tới chỗ cuối)
    return first, second

for arr in ([1, 2, 6, 4, 5, 3, 7], [1, 2, 4, 3, 5, 6, 7], [3, 2, 1], [2, 1]):
    f, s = find_swapped(arr)
    fixed = arr[:]
    fixed[f], fixed[s] = fixed[s], fixed[f]
    print(f"{arr} -> đổi vị trí {f} ({arr[f]}) và {s} ({arr[s]}) -> {fixed}")

[1, 2, 6, 4, 5, 3, 7] -> đổi vị trí 2 (6) và 5 (3) -> [1, 2, 3, 4, 5, 6, 7]
[1, 2, 4, 3, 5, 6, 7] -> đổi vị trí 2 (4) và 3 (3) -> [1, 2, 3, 4, 5, 6, 7]
[3, 2, 1] -> đổi vị trí 0 (3) và 2 (1) -> [1, 2, 3]
[2, 1] -> đổi vị trí 0 (2) và 1 (1) -> [1, 2]


# 5. Baseline: lấy inorder ra mảng, sắp xếp, ghi lại

Lấy danh sách **nút** theo inorder, sắp xếp các **giá trị**, gán lại lần lượt. `O(n log n)` thời gian, `O(n)` bộ nhớ — đúng nhưng chưa tận dụng "chỉ có hai nút sai".

In [11]:
def recover_sort(root: Optional[TreeNode]) -> None:
    nodes = []
    def dfs(node):
        if node:
            dfs(node.left); nodes.append(node); dfs(node.right)
    dfs(root)
    for node, v in zip(nodes, sorted(n.val for n in nodes)):
        node.val = v

t = build_tree([3, 1, 4, None, None, 2])
recover_sort(t)
print(serialize(t))

[2, 1, 4, None, None, 3]


# 6. Lời giải chính: inorder bằng stack, theo dõi `prev`

Duyệt inorder (như #98), giữ `prev` (nút được thăm ngay trước). Mỗi khi `prev.val > node.val`:

- nếu chưa có `first`: `first = prev`
- luôn cập nhật `second = node`

Kết thúc: đổi `first.val` và `second.val`.

In [12]:
def recover_stack(root: Optional[TreeNode]) -> None:
    stack, node = [], root
    prev = first = second = None
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        if prev and prev.val > node.val:
            if first is None:
                first = prev
            second = node
        prev = node
        node = node.right
    first.val, second.val = second.val, first.val

for vals in ([1, 3, None, None, 2], [3, 1, 4, None, None, 2]):
    t = build_tree(vals)
    recover_stack(t)
    print(vals, "->", serialize(t))

[1, 3, None, None, 2] -> [3, 1, None, None, 2]
[3, 1, 4, None, None, 2] -> [2, 1, 4, None, None, 3]


# 7. Mô phỏng

In [13]:
def trace_recover(values: list) -> list:
    root = build_tree(values)
    stack, node = [], root
    prev = first = second = None
    print("inorder trước:", inorder_list(root))
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        msg = ""
        if prev and prev.val > node.val:
            if first is None:
                first = prev
                msg = f"  <- chỗ giảm ĐẦU: first = {first.val}, second = {node.val}"
            else:
                msg = f"  <- chỗ giảm THỨ HAI: second = {node.val}"
            second = node
        print(f"  thăm {node.val} (prev = {prev.val if prev else None}){msg}")
        prev = node
        node = node.right
    first.val, second.val = second.val, first.val
    print(f"Đổi {second.val} <-> {first.val}. inorder sau:", inorder_list(root))
    return serialize(root)

_ = trace_recover([3, 1, 4, None, None, 2])
print()
_ = trace_recover([4, 2, 6, 1, 7, 5, 3])

inorder trước: [1, 3, 2, 4]
  thăm 1 (prev = None)
  thăm 3 (prev = 1)
  thăm 2 (prev = 3)  <- chỗ giảm ĐẦU: first = 3, second = 2
  thăm 4 (prev = 2)
Đổi 3 <-> 2. inorder sau: [1, 2, 3, 4]

inorder trước: [1, 2, 7, 4, 5, 6, 3]
  thăm 1 (prev = None)
  thăm 2 (prev = 1)
  thăm 7 (prev = 2)
  thăm 4 (prev = 7)  <- chỗ giảm ĐẦU: first = 7, second = 4
  thăm 5 (prev = 4)
  thăm 6 (prev = 5)
  thăm 3 (prev = 6)  <- chỗ giảm THỨ HAI: second = 3
Đổi 7 <-> 3. inorder sau: [1, 2, 3, 4, 5, 6, 7]


| Ví dụ | Hai nút bị đổi | Số chỗ giảm | `first`, `second` |
|---|---|---|---|
| `[3,1,4,null,null,2]` | `2` và `3` — **kề nhau** trong inorder | 1: `(3, 2)` | cả hai phần tử của chỗ giảm đó |
| `[4,2,6,1,7,5,3]` (BST `1..7` với `3` và `7` bị đổi) | **xa nhau** | 2: `(7, 4)` và `(6, 3)` | `7` (lớn của chỗ 1) và `3` (nhỏ của chỗ 2) |

# 8. Follow-up: Morris traversal — `O(1)` bộ nhớ

Stack tốn `O(h)`. **Morris** duyệt inorder mà không cần stack, bằng cách **tạm thời** nối nút lớn nhất của cây con trái (predecessor) về nút hiện tại ("sợi chỉ", thread), rồi gỡ ra sau khi dùng:

```text
cur = root
while cur:
    if cur.left is None:
        thăm cur; cur = cur.right
    else:
        pre = nút phải nhất của cur.left (dừng nếu pre.right == cur)
        if pre.right is None:        # lần đầu tới cur: nối sợi chỉ, đi sang trái
            pre.right = cur; cur = cur.left
        else:                        # lần hai (quay về qua sợi chỉ): gỡ sợi chỉ, thăm cur
            pre.right = None; thăm cur; cur = cur.right
```

Mỗi cạnh được đi qua tối đa vài lần → `O(n)` thời gian, `O(1)` bộ nhớ phụ.

In [14]:
def morris_inorder(root: Optional[TreeNode]):
    cur = root
    while cur:
        if cur.left is None:
            yield cur
            cur = cur.right
        else:
            pre = cur.left
            while pre.right and pre.right is not cur:
                pre = pre.right
            if pre.right is None:
                pre.right = cur
                cur = cur.left
            else:
                pre.right = None
                yield cur
                cur = cur.right


def recover_morris(root: Optional[TreeNode]) -> None:
    prev = first = second = None
    for node in morris_inorder(root):
        if prev and prev.val > node.val:
            if first is None:
                first = prev
            second = node
        prev = node
    first.val, second.val = second.val, first.val

t = build_tree([3, 1, 4, None, None, 2])
print("Morris inorder:", [n.val for n in morris_inorder(t)])
recover_morris(t)
print("Sau khi khôi phục:", serialize(t))

Morris inorder: [1, 3, 2, 4]
Sau khi khôi phục: [2, 1, 4, None, None, 3]


Chú ý: generator `morris_inorder` phải được chạy **hết** để mọi sợi chỉ được gỡ — nếu dừng giữa chừng (`break`), cây sẽ còn các con trỏ tạm và bị hỏng cấu trúc.

# 9. Phiên bản đúng format LeetCode

Dùng Morris để đạt follow-up.

In [15]:
class Solution:
    def recoverTree(self, root: Optional[TreeNode]) -> None:
        prev = first = second = None
        cur = root
        while cur:
            if cur.left is None:
                visit = cur
                cur = cur.right
            else:
                pre = cur.left
                while pre.right and pre.right is not cur:
                    pre = pre.right
                if pre.right is None:
                    pre.right = cur
                    cur = cur.left
                    continue
                pre.right = None
                visit = cur
                cur = cur.right
            if prev and prev.val > visit.val:
                if first is None:
                    first = prev
                second = visit
            prev = visit
        first.val, second.val = second.val, first.val

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<10} PASS  {actual}")


def run(f, values):
    t = build_tree(values)
    f(t)
    return serialize(t)


def shape(t):
    return None if not t else (shape(t.left), shape(t.right))


def random_bst_values(n):
    """Tạo một BST ngẫu nhiên với giá trị phân biệt, trả về gốc."""
    vals = random.sample(range(-50, 50), n)
    root = None
    def insert(node, v):
        if not node:
            return TreeNode(v)
        if v < node.val:
            node.left = insert(node.left, v)
        else:
            node.right = insert(node.right, v)
        return node
    for v in vals:
        root = insert(root, v)
    return root


sol = Solution()
check("Ví dụ 1", run(sol.recoverTree, [1, 3, None, None, 2]), [3, 1, None, None, 2])
check("Ví dụ 2", run(sol.recoverTree, [3, 1, 4, None, None, 2]), [2, 1, 4, None, None, 3])
check("Hai nút", run(sol.recoverTree, [1, 2]), [2, 1])

random.seed(99)
for _ in range(3000):
    good = random_bst_values(random.randint(2, 12))
    correct = serialize(good)
    nodes = []
    def collect(nd):
        if nd:
            collect(nd.left); nodes.append(nd); collect(nd.right)
    collect(good)
    a, b = random.sample(nodes, 2)
    a.val, b.val = b.val, a.val                      # làm hỏng: đổi hai nút
    broken = serialize(good)
    for f in (sol.recoverTree, recover_stack, recover_morris, recover_sort):
        t = build_tree(broken)
        before_shape = shape(t)
        assert f(t) is None
        assert serialize(t) == correct and shape(t) == before_shape   # đúng giá trị, cấu trúc không đổi
print("3000 BST bị đổi hai nút ngẫu nhiên: 4 cách đều khôi phục đúng, không đổi cấu trúc")

Ví dụ 1    PASS  [3, 1, None, None, 2]


Ví dụ 2    PASS  [2, 1, 4, None, None, 3]
Hai nút    PASS  [2, 1]


3000 BST bị đổi hai nút ngẫu nhiên: 4 cách đều khôi phục đúng, không đổi cấu trúc

# 10. Vì sao đúng? Độ phức tạp?

Gọi dãy inorder đúng là `a_0 < a_1 < … < a_{n−1}`, và hai vị trí bị đổi là `p < q`.

- Nếu `q = p + 1`: chỉ cặp `(p, p+1)` bị giảm → `first` = vị trí `p`, `second` = vị trí `p+1`.
- Nếu `q > p + 1`: giá trị lớn `a_q` đứng ở `p` nên `(p, p+1)` giảm; giá trị nhỏ `a_p` đứng ở `q` nên `(q−1, q)` giảm; các cặp khác vẫn tăng. → `first` = vị trí `p` (chỗ giảm đầu), `second` = vị trí `q` (chỗ giảm cuối).

Cả hai trường hợp, quy tắc "`first` = lớn của chỗ giảm đầu, `second` = nhỏ của chỗ giảm cuối" đều chọn đúng.

```text
Time:  O(n)
Space: O(h) với stack | O(1) với Morris
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Dừng ở chỗ giảm đầu tiên

In [17]:
def wrong_stop_first(root: Optional[TreeNode]) -> None:
    stack, node = [], root
    prev = None
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        if prev and prev.val > node.val:
            prev.val, node.val = node.val, prev.val    # SAI: đổi ngay chỗ giảm đầu
            return
        prev = node
        node = node.right

t = build_tree([4, 2, 6, 1, 7, 5, 3])                  # BST [4,2,6,1,3,5,7] với 3 và 7 bị đổi
print("inorder hỏng:", inorder_list(t))
wrong_stop_first(t)
print("Sai : inorder sau khi sửa", inorder_list(t))
t = build_tree([4, 2, 6, 1, 7, 5, 3]); sol.recoverTree(t)
print("Đúng: inorder sau khi sửa", inorder_list(t))

inorder hỏng: [1, 2, 7, 4, 5, 6, 3]
Sai : inorder sau khi sửa [1, 2, 4, 7, 5, 6, 3]
Đúng: inorder sau khi sửa [1, 2, 3, 4, 5, 6, 7]


Khi hai nút **xa nhau**, chỗ giảm đầu là `(7, 4)` — phần tử nhỏ `4` không phải nút sai. Phải đợi tới chỗ giảm thứ hai `(6, 3)` mới biết nút sai thứ hai là `3`.

## Lỗi 2: Lấy `second` là phần tử lớn của chỗ giảm thứ hai

`second` là phần tử **nhỏ** (phần tử sau) của chỗ giảm cuối.

## Lỗi 3: Đổi cả nút (con trỏ) thay vì đổi giá trị

Đề chỉ yêu cầu đổi lại giá trị; đổi nút phải sửa con trỏ cha/con — phức tạp và dễ sai.

## Lỗi 4: Morris — quên gỡ sợi chỉ

Nếu không đặt `pre.right = None` ở lần thứ hai, cây còn vòng lặp và duyệt sau sẽ chạy mãi.

# 12. Mẹo nhận diện pattern

- **"BST gần đúng"** → nhìn vào inorder và tìm chỗ phá vỡ tính tăng dần.
- **"Đổi chỗ đúng hai phần tử trong dãy đã sắp xếp"** → quy tắc chỗ giảm đầu / chỗ giảm cuối (dùng được cả cho mảng).
- **Duyệt cây `O(1)` bộ nhớ** → Morris traversal.

Bài liên quan: **#98 Validate BST**, **#94 Binary Tree Inorder Traversal** (Morris), **#538 Convert BST to Greater Tree** (inorder ngược), **#1305 All Elements in Two BSTs**.

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Dãy inorder `[1, 5, 3, 4, 2, 6]`: chỉ ra các chỗ giảm, `first`, `second`.

## Bài 2 — Điền chỗ trống (bản stack)

In [18]:
def exercise_2_recover_tree(root: Optional[TreeNode]) -> None:
    stack, node = [], root
    prev = first = second = None
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        # TODO: phát hiện chỗ giảm, cập nhật first (một lần) và second (mỗi lần)
        prev = node
        node = node.right
    # TODO: đổi giá trị first và second


# t = build_tree([1, 3, None, None, 2]); exercise_2_recover_tree(t); assert serialize(t) == [3, 1, None, None, 2]
# t = build_tree([3, 1, 4, None, None, 2]); exercise_2_recover_tree(t); assert serialize(t) == [2, 1, 4, None, None, 3]

## Bài 3 — Biến thể: trên mảng

Mảng `nums` vốn tăng nghiêm ngặt nhưng **đúng hai** phần tử bị đổi chỗ. Khôi phục tại chỗ trong `O(n)`.

In [19]:
def exercise_3_recover_array(nums: List[int]) -> None:
    # TODO
    pass


# a = [1, 2, 6, 4, 5, 3, 7]; exercise_3_recover_array(a); assert a == [1, 2, 3, 4, 5, 6, 7]
# a = [2, 1]; exercise_3_recover_array(a); assert a == [1, 2]

## Bài 4 — Biến thể: Binary Tree Inorder Traversal (LeetCode #94) bằng Morris

Trả list giá trị inorder, `O(1)` bộ nhớ phụ (không tính output), và cây phải **nguyên vẹn** sau khi duyệt.

In [20]:
def exercise_4_inorder_morris(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# t = build_tree([1, None, 2, 3]); assert exercise_4_inorder_morris(t) == [1, 3, 2] and serialize(t) == [1, None, 2, 3]
# assert exercise_4_inorder_morris(None) == []

## Bài 5 — Mini challenge: đếm số "nghịch thế" giữa hai nút

Cho cây nhị phân (không nhất thiết BST), đếm số cặp nút `(u, v)` với `u` đứng **trước** `v` trong inorder nhưng `u.val > v.val` — tức số nghịch thế của dãy inorder. `O(n log n)`.

Gợi ý nhanh: lấy inorder rồi đếm nghịch thế bằng **merge sort**.

In [21]:
def exercise_5_count_inversions(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_5_count_inversions(build_tree([3, 1, 4, None, None, 2])) == 1
# assert exercise_5_count_inversions(build_tree([2, 1, 3])) == 0
# assert exercise_5_count_inversions(build_tree([1, 2, 3])) == 1

# 14. Gợi ý

**Bài 1**
- Mức 1: chỗ giảm: `(5, 3)` và `(4, 2)`.

**Bài 3**
- Mức 1: dùng `find_swapped` ở mục 4, rồi đổi hai vị trí.

**Bài 4**
- Mức 1: chép khung `morris_inorder`, thay `yield` bằng `out.append(cur.val)`.

**Bài 5**
- Mức 1: `sort_count(a)`: chia đôi, đếm đệ quy hai nửa, khi trộn: mỗi lần lấy từ nửa phải, cộng số phần tử **còn lại** của nửa trái.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
arr = [1, 5, 3, 4, 2, 6]
f, s = find_swapped(arr)
print("chỗ giảm:", [(arr[i], arr[i + 1]) for i in range(len(arr) - 1) if arr[i] > arr[i + 1]], "| first =", arr[f], "| second =", arr[s])

chỗ giảm: [(5, 3), (4, 2)] | first = 5 | second = 2


## Đáp án Bài 2

In [23]:
def answer_2_recover_tree(root: Optional[TreeNode]) -> None:
    recover_stack(root)

t = build_tree([1, 3, None, None, 2]); answer_2_recover_tree(t); assert serialize(t) == [3, 1, None, None, 2]
t = build_tree([3, 1, 4, None, None, 2]); answer_2_recover_tree(t); assert serialize(t) == [2, 1, 4, None, None, 3]
print("Bài 2 PASS (giống recover_stack ở mục 6)")

Bài 2 PASS (giống recover_stack ở mục 6)


## Đáp án Bài 3

In [24]:
def answer_3_recover_array(nums: List[int]) -> None:
    f, s = find_swapped(nums)
    nums[f], nums[s] = nums[s], nums[f]

a = [1, 2, 6, 4, 5, 3, 7]; answer_3_recover_array(a); assert a == [1, 2, 3, 4, 5, 6, 7]
a = [2, 1]; answer_3_recover_array(a); assert a == [1, 2]
for _ in range(3000):
    n = random.randint(2, 10)
    good = sorted(random.sample(range(100), n))
    a = good[:]
    i, j = random.sample(range(n), 2)
    a[i], a[j] = a[j], a[i]
    answer_3_recover_array(a)
    assert a == good
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [25]:
def answer_4_inorder_morris(root: Optional[TreeNode]) -> List[int]:
    return [node.val for node in morris_inorder(root)]

t = build_tree([1, None, 2, 3]); assert answer_4_inorder_morris(t) == [1, 3, 2] and serialize(t) == [1, None, 2, 3]
assert answer_4_inorder_morris(None) == []
for _ in range(1000):
    t = random_bst_values(random.randint(1, 10))
    before = serialize(t)
    assert answer_4_inorder_morris(t) == inorder_list(t) and serialize(t) == before
print("Bài 4 PASS (cây nguyên vẹn sau khi duyệt)")

Bài 4 PASS (cây nguyên vẹn sau khi duyệt)


## Đáp án Bài 5

In [26]:
def answer_5_count_inversions(root: Optional[TreeNode]) -> int:
    def sort_count(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        left, x = sort_count(a[:mid])
        right, y = sort_count(a[mid:])
        merged, inv, i, j = [], x + y, 0, 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
                inv += len(left) - i
        merged += left[i:] + right[j:]
        return merged, inv
    return sort_count(inorder_list(root))[1]

assert answer_5_count_inversions(build_tree([3, 1, 4, None, None, 2])) == 1
assert answer_5_count_inversions(build_tree([2, 1, 3])) == 0
assert answer_5_count_inversions(build_tree([1, 2, 3])) == 1
for _ in range(1000):
    vals = [random.randint(0, 9) for _ in range(random.randint(1, 10))]
    t = build_tree(vals)
    seq = inorder_list(t)
    assert answer_5_count_inversions(t) == sum(seq[i] > seq[j] for i in range(len(seq)) for j in range(i + 1, len(seq)))
print("Bài 5 PASS")

Bài 5 PASS


Liên hệ với bài chính: BST bị đổi đúng hai nút **kề nhau** trong inorder có đúng 1 nghịch thế; đổi hai nút xa nhau cách `d` vị trí thì có `2d − 1` nghịch thế.

# 16. Tổng kết

> Duyệt inorder, giữ `prev`. Mỗi khi `prev.val > node.val`: lần đầu đặt `first = prev`; mọi lần đặt `second = node`. Cuối cùng đổi giá trị `first` và `second`. Stack `O(h)`, Morris `O(1)`.

---

## Checklist trước khi nộp

- [ ] Không dừng ở chỗ giảm đầu tiên?
- [ ] `first` là phần tử **lớn** của chỗ giảm đầu, `second` là phần tử **nhỏ** của chỗ giảm cuối?
- [ ] Đổi **giá trị**, không đổi cấu trúc?
- [ ] Morris: gỡ mọi sợi chỉ?